In [2]:
# Cell 1: 通用的模型调用
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="MiniMax-M3",
    api_key="sk-cp-GsTRU_GlFRX_WPY8cf8IqBTDtrVJP4zl4EWLZ8D7627lrDtV-MbWYnWWK7rmvT7ZRluuMtUL7VP2sCD2uq6SzkpIlY0EAJtCIHW2uSGAT4QKYxX9mJQmK8I",
    base_url="https://api.minimaxi.com/v1",
    temperature=0,
)

In [6]:
# Cell 2: 定义 Pydantic 数据模型
from pydantic import BaseModel, Field

class Person(BaseModel):
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")
    hobby: str = Field(description="爱好")

In [7]:
# Cell 3: 工具函数 —— 清理模型输出
import re

def strip_think_and_fence(text: str) -> str:
    """从模型输出中提取纯 JSON 字符串。"""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    if m:
        return m.group(1)
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end != -1 and end > start:
        return text[start:end + 1]
    return text

In [13]:
# Cell 4: 搭建 Chain = Prompt → Model → 清洗 → Pydantic 解析
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=Person)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "你是信息提取助手。只输出合法 JSON，禁止思考、寒暄、markdown。\n"
     "{format_instructions}"),
    ("human", "{input}"),
]).partial(format_instructions=parser.get_format_instructions())

def clean(msg) -> str:
    content = msg.content if hasattr(msg, "content") else msg
    return strip_think_and_fence(content)

chain = prompt | model | clean | parser
print("Chain 构建完成 ✅")

Chain 构建完成 ✅


In [ ]:
# Cell 5: 调用 Chain
response = chain.invoke({"input": "张三是一名30岁的软件工程师,爱好是打篮球。请提取他的姓名、年龄、职业和爱好。"})

print("=== 结构化结果 ===")
print(type(response).__name__, ":", response)
print()
print("=== 字段访问 ===")
print("姓名:", response.name)
print("年龄:", response.age)
print("职业:", response.occupation)
print("爱好:", response.hobby)

=== 结构化结果 ===
Person : name='张三' age=30 occupation='软件工程师' hobby='打篮球'

=== 字段访问 ===
姓名: 张三
年龄: 30
职业: 软件工程师
